# Macro Distance Model for Sector Rotation

Same methodoloty as factor timing, but on sectors.

## Methodology

1. **7 financial state variables** (oil, copper, vix, yield curve, monetary policy, market, stock-bond correlation)
2. **Normalise** each variable via rolling z-score (10-year window), winsorised at +/-3
3. **Euclidean distance** between current month and all historical months
4. **Exclude last 36 months** from similarity calculation to avioid momentum loading
5. **Forecast factor returns** using similarity-weighted historical averages
6. **Backtest** against Fama-French 6 factors + Momentum

### Sector Universe
| ETF | Sector / Industry Group |
|---|---|
| XLK | Technology |
| XLF | Financials |
| XLV | Health Care |
| XLY | Consumer Discretionary |
| XLP | Consumer Staples |
| XLE | Energy |
| XLI | Industrials |
| XLB | Materials |
| XLU | Utilities |
| XLRE | Real Estate |
| XLC | Communication Services |
| XBI | Biotech |
| XHB | Homebuilders |
| XOP | Oil & Gas E&P |
| SMH | Semiconductors |
| GLD | Gold |
| TLT | LT Bond |

* This version forecasts **long-only sector excess returns** (vs equal-weighted benchmark), producing a sector rotation signal rather than a factor tilt signal.

In [1]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly.io as pio
pio.renderers.default = 'notebook'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_row', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

COLOURS = sns.color_palette('muted').as_hex()
CONTRAST = [
    '#0675C2', '#F7941E', '#66C7EC', '#F9A94B', '#6CC263',
    '#A7DAA1', '#333333', '#999999', '#C5C5C7'
]

PLOTLY_LAYOUT = dict(
    template='plotly_white',
    font=dict(size=10),
    title_font=12,
    hovermode='x unified',
    autosize=True,
    margin=dict(l=60, r=30, t=60, b=80)
)

LEGEND_BELOW = dict(
    orientation='h', yanchor='top', y=-0.15, xanchor='left', x=0, font=dict(size=10)
)


## 1. Load State Variables

Source the 7 financial indicators from public data. All variables are converted to 12-month changes.

| Variables | Source | Ticker |
|---|---|---|
| Oil price | Yahoo Finance | `BZ=F` |
| Copper price | Yahoo Finance | `HG=F` |
| US equity vol | Yahoo Finance | `^VIX` |
| Yield curve | 10Y yield minus 3m yield | derived |
| Monetary policy | 3m yield | `^IRX` |
| Market | S&P 500 | `^SPX` |
| Stock bond correlation | rolling 12M corr(S&P 500, -10Y) | derived |

In [78]:
# ==========================================================================
# Configuration
# ==========================================================================

# Distance model parameters
DISTANCE_CONFIG = dict(
    norm_window=120,
    winsorize_limit=3.0,
    exclusion_window=36,
    min_periods=24
)

# Sector / industry group ETFs
SECTOR_ETFS = {
    'XLK': 'Technology',
    'XLF': 'Financials',
    'XLV': 'Health Care',
    'XLY': 'Consumer Disc.',
    'XLP': 'Consumer Staples',
    'XLE': 'Energy',
    'XLI': 'Industrials',
    'XLB': 'Materials',
    'XLU': 'Utilities',
    'XLRE': 'Real Estate',
    'XLC': 'Comm. Services',
    # 'XBI': 'Biotech',
    # 'XHB': 'Homebuilders',
    # 'XOP': 'Oil & Gas E&P',
    # 'SMH': 'Semiconductors',
    'GLD': 'Gold',
    'TLT': 'LT Bond',
}

DATA_START = '1999-01-01'

print("Configuration loaded")
print(f"Distance config:  {DISTANCE_CONFIG}")

Configuration loaded
Distance config:  {'norm_window': 120, 'winsorize_limit': 3.0, 'exclusion_window': 36, 'min_periods': 24}


In [3]:
# ==========================================================================
# Load state variables from sources
# ==========================================================================

yf_tickers = {
    'oil': 'BZ=F',
    'copper': 'HG=F',
    'vix': '^VIX',
    'us_10y': '^TNX',
    'us_3m' : '^IRX',
    'spx' : '^SPX'
}

print('Downloading from Yahoo Finance data...')
yf_data = {}
for name, ticker in yf_tickers.items():
    try:
        df = yf.download(ticker, start=DATA_START, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            series = df['Close'].iloc[:, 0]
        else:
            series = df['Close']
        series.index = pd.to_datetime(series.index)
        yf_data[name] = series.dropna()
    except Exception as e:
        print(f'  {name} ({ticker}): FAILED - {e}')

fred_series = {
    'credit_spread': 'BAMLC0A4CBBB',    # ICE BofA BBB US Corporate Index
    'cpi': 'CPIAUCSL'                   # CPI for all urban consumers
}

print('\nDownloading FRED data...')
for name, series_id in fred_series.items():
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd=1990-01-01'
    try:
        df = pd.read_csv(url, parse_dates=['observation_date'], index_col='observation_date')
        df.columns = ['value']
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        yf_data[name] = df['value'].dropna()
    except Exception as e:
        print(f'  {name} ({series_id}: FAILED - {e})')

print(f'\nLoaded {len(yf_data)} series: {list(yf_data.keys())}')



Loaded 8 series: ['oil', 'copper', 'vix', 'us_10y', 'us_3m', 'spx', 'credit_spread', 'cpi']


In [92]:
# ==========================================================================
# Transform state variable
# ==========================================================================

# Resample to month end
macro_monthly = pd.DataFrame()
for name, series in yf_data.items():
    monthly = series.resample('ME').last()
    macro_monthly[name] = monthly

state_vars = pd.DataFrame(index=macro_monthly.index)

# Price like variable as % change
for var in ['oil', 'copper', 'vix', 'spx']:
    if var in macro_monthly.columns:
        state_vars[var] = macro_monthly[var].pct_change(12)

# Rate / Yield as absolute change
for var in ['credit_spread', 'us_10y', 'us_3m']:
    if var in macro_monthly.columns:
        state_vars[var] = macro_monthly[var].diff(12)

# Real rates = 10Y yield minus YoY CPI inflation
if 'us_10y' in macro_monthly.columns and 'cpi' in macro_monthly.columns:
    cpi_yoy = macro_monthly['cpi'].pct_change(12) * 100
    real_rate = macro_monthly['us_10y'] - cpi_yoy
    state_vars['real_rates'] = real_rate.diff(12)
    print(f'Real rate derived: {state_vars['real_rates'].dropna().shape[0]} months')

# Yield curve
if 'us_10y' in macro_monthly.columns and 'us_3m' in macro_monthly.columns:
    yield_curve = macro_monthly['us_10y'] - macro_monthly['us_3m']
    state_vars['yield_curve'] = yield_curve.diff(12)
    print(f'Yield curve derived: {state_vars['yield_curve'].dropna().shape[0]} months')

# Stock bond correlation
if 'us_10y' in macro_monthly.columns and 'spx' in macro_monthly.columns:
    market_ret = macro_monthly['spx'].pct_change()
    bond_ret = -macro_monthly['us_10y'].diff()  # Bond return proxy: negative of yield change
    stock_bond_corr = market_ret.rolling(12).corr(bond_ret)
    state_vars['stock_bond_corr'] = stock_bond_corr.diff(12)

state_vars = state_vars.dropna(how='all')
print(f'Date range: {state_vars.index.min().date()} to {state_vars.index.max().date()}')
print('Summary:')
for col in state_vars.columns:
    s = state_vars[col].dropna()
    print(f'  {col:15s}: n={len(s):4d}, mean={s.mean():8.4f}, std={s.std():8.4f}, min={s.min():8.4f}, max={s.max():8.4f}')

# print(f'\nMissing data (%): {state_vars.isna().mean() * 100:.1f}')

Real rate derived: 199 months
Yield curve derived: 226 months
Date range: 2007-07-31 to 2026-04-30
Summary:
  oil            : n= 214, mean=  0.0508, std=  0.3880, min= -0.6675, max=  1.7942
  copper         : n= 214, mean=  0.0597, std=  0.2981, min= -0.6038, max=  1.3853
  vix            : n= 214, mean=  0.0857, std=  0.5139, min= -0.6377, max=  2.9052
  spx            : n= 214, mean=  0.1037, std=  0.1630, min= -0.4476, max=  0.5371
  credit_spread  : n= 214, mean= -0.0816, std=  1.3601, min= -5.3600, max=  5.3900
  us_10y         : n= 214, mean=  0.0082, std=  0.8540, min= -1.8870, max=  2.5200
  us_3m          : n= 214, mean=  0.0632, std=  1.2895, min= -3.3850, max=  4.4220
  real_rates     : n= 199, mean= -0.0152, std=  2.2592, min= -5.8885, max=  6.9783
  yield_curve    : n= 226, mean=  1.3506, std=  1.2385, min= -1.6110, max=  3.7930
  stock_bond_corr: n= 214, mean= -0.1219, std=  0.4583, min= -0.8514, max=  0.8108


## 2. Load Sector & Industry Group ETF Returns

Download monthly total returns for sector/industry ETF from Yahoo Finance.
ETFs with shorter histories are included where available.

In [93]:
# ==========================================================================
# Load sector / industry ETF returns
# ==========================================================================

print("Downloading sector/industry ETF prices...")
sector_prices = pd.DataFrame()
failed_tickers = []

for ticker, label in SECTOR_ETFS.items():
    try:
        df = yf.download(ticker, start=DATA_START, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            series = df['Close'].iloc[:, 0]
        else:
            series = df['Close']
        series.index = pd.to_datetime(series.index)
        monthly = series.resample('ME').last()
        sector_prices[ticker] = monthly
        start_dt = monthly.dropna().index.min().date()
        print(f'  {ticker:5s} ({label:20s}): from {start_dt}, {monthly.dropna().shape[0]} months')
    except Exception as e:
        print(f'  {ticker:5s} ({label:20s}): Failed - {e}')
        failed_tickers.append(ticker)

# Remove failed tickers from universe
for t in failed_tickers:
    SECTOR_ETFS.pop(t, None)

# Monthly return
sector_returns = sector_prices.pct_change()

# SPY as market benchmark
print('\nDownloading SPY...')
spy = yf.download('SPY', start=DATA_START, progress=False, auto_adjust=True)
if isinstance(spy.columns, pd.MultiIndex):
    spy_close = spy['Close'].iloc[:, 0]
else:
    spy_close = spy['Close']
spy_monthly = spy_close.resample('ME').last()
spy_returns = spy_monthly.pct_change()
spy_returns.name = 'SPY'

print(f'\nSector return matrix: {sector_returns.shape}')
print(f'Date range: {sector_returns.index.min().date()} to {sector_returns.index.max().date()}')

# Summary
summary = []
for ticker in sector_returns.columns:
    s = sector_returns[ticker].dropna()
    ann_ret = s.mean() * 12
    ann_vol = s.std() * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    summary.append({
        'ETF': ticker, 'Sector': SECTOR_ETFS.get(ticker, ''),
        'Start': s.index.min().strftime('%Y-%m'), 'Months': len(s),
        'Ann. Ret (%)': round(ann_ret * 100, 2),
        'Ann. Vol (%)': round(ann_vol * 100, 2),
        'Sharpe': round(sharpe, 2)
    })

summary_df = pd.DataFrame(summary).sort_values('Sharpe', ascending=False)
print('\nSector ETF Summary:')
print(summary_df.to_string(index=False))

  XLK   (Technology          ): from 1999-01-31, 328 months
  XLF   (Financials          ): from 1999-01-31, 328 months
  XLV   (Health Care         ): from 1999-01-31, 328 months
  XLY   (Consumer Disc.      ): from 1999-01-31, 328 months
  XLP   (Consumer Staples    ): from 1999-01-31, 328 months
  XLE   (Energy              ): from 1999-01-31, 328 months
  XLI   (Industrials         ): from 1999-01-31, 328 months
  XLB   (Materials           ): from 1999-01-31, 328 months
  XLU   (Utilities           ): from 1999-01-31, 328 months
  XLRE  (Real Estate         ): from 2015-10-31, 127 months
  XLC   (Comm. Services      ): from 2018-06-30, 95 months
  GLD   (Gold                ): from 2004-11-30, 258 months
  TLT   (LT Bond             ): from 2002-07-31, 286 months


Sector return matrix: (328, 13)
Date range: 1999-01-31 to 2026-04-30

Sector ETF Summary:
 ETF           Sector   Start  Months  Ann. Ret (%)  Ann. Vol (%)  Sharpe
 GLD             Gold 2004-12     257       11.9800    

In [80]:
# ==========================================================================
# Add equity dispersion to state variables
# ==========================================================================
# Use cross-sectional standard deviation of the 11 GICS sector ETFs as the
# dispersion measure.

core_sectors = ['XLK', 'XLF', 'XLV', 'XLY', 'XLP', 'XLE', 'XLI', 'XLB', 'XLU']
available_core = [t for t in core_sectors if t in sector_returns.columns]

dispersion = sector_returns[available_core].std(axis=1)
dispersion.name = 'dispersion'

# 12 month absolute change in dispersion
state_vars['dispersion'] = dispersion.diff(12)
state_vars = state_vars.dropna(how='all')

# ==========================================================================
# Dispersion time series
# ==========================================================================
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=(
        'Sector Dispersion (Cross-Sectional Std Dev of Monthly Returns)',
        'Dispersion 12-month Change'
    )
)

# Raw level
disp_level = dispersion.dropna()
fig.add_trace(go.Scatter(
    x=disp_level.index, y=disp_level.values,
    name='Dispersion (level)',
    line=dict(color=COLOURS[0], width=1.5)
), row=1, col=1)

# 12 month change
disp_change = state_vars['dispersion'].dropna()
fig.add_trace(go.Scatter(
    x=disp_change.index, y=disp_change.values,
    name='12M change',
    line=dict(color=COLOURS[1], width=1.5)
), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color=COLOURS[3], row=2, col=1)

fig.update_yaxes(title_text='Std Dev', row=1, col=1)
fig.update_yaxes(title_text='12M Change Std Dev', row=2, col=1)
fig.update_layout(
    height=800,
    template='plotly_white',
    showlegend=False
)
fig.show()

## 3. Compute Excess Returns

For sector rotation, the relevant signal is **excess return vs the equal-weighted sector benchmark**.
E.g. A sector that returns 5% when the average sector returns 8% underperformed.

We compute:
$$r^{xs}_{i,t} = r_{i,t} - \bar{r}_t$$

where $\bar{r}_t$ is the cross-sectional mean return across all available sectors at time $t$.

In [81]:
# ==========================================================================
# Compute excess return (vs equal weighted benchmark)
# ==========================================================================

ew_benchmark = sector_returns.mean(axis=1)
ew_benchmark.name = 'EW_Benchmark'
# spy_benchmark = spy_returns

excess_returns = sector_returns.sub(ew_benchmark, axis=0)

print(f'Mean absolute row sum: {excess_returns.sum(axis=1).abs().mean():.6f}')

# Align state variables and excess returns
common_dates = state_vars.index.intersection(excess_returns.index)
print(f'\nCommon date range: {common_dates.min().date()} to {common_dates.max().date()}')
print(f'Common months: {len(common_dates)}')

state_aligned = state_vars.loc[common_dates].copy()
excess_aligned = excess_returns.loc[common_dates].copy()
sector_aligned = sector_returns.loc[common_dates].copy()

# Drop any state variables with > 50% missing
missing_pct = state_aligned.isna().mean()
good_vars = missing_pct[missing_pct < 0.5].index.tolist()
dropped_vars = missing_pct[missing_pct >= 0.5].index.tolist()
state_aligned = state_aligned[good_vars]

if dropped_vars:
    print(f'\n Dropped variables (>50% missing): {dropped_vars}')

print(f'\nState variables retained: {len(good_vars)}:')
for v in good_vars:
    n_valid = state_aligned[v].notna().mean()
    pct_miss = missing_pct[v] * 100
    print(f'   {v:15s}: {n_valid} obs, {pct_miss:.1f}% missing')

# Forward fill gaps of up to 3 months
print('\nForward fill missing (up to 3 months)...')
state_aligned = state_aligned.ffill(limit=3)

Mean absolute row sum: 0.000000

Common date range: 2008-07-31 to 2026-04-30
Common months: 214

State variables retained: 11:
   oil            : 1.0 obs, 0.0% missing
   copper         : 1.0 obs, 0.0% missing
   vix            : 1.0 obs, 0.0% missing
   spx            : 1.0 obs, 0.0% missing
   credit_spread  : 1.0 obs, 0.0% missing
   us_10y         : 1.0 obs, 0.0% missing
   us_3m          : 1.0 obs, 0.0% missing
   real_rates     : 0.9299065420560748 obs, 7.0% missing
   yield_curve    : 1.0 obs, 0.0% missing
   stock_bond_corr: 0.9439252336448598 obs, 5.6% missing
   dispersion     : 1.0 obs, 0.0% missing

Forward fill missing (up to 3 months)...


## 3. Fit Distance Model

Implement regime similarity model:
- Rolling 10 year z-score normalization (min 2 year)
- Winsorisation at +/-3
- 36 month exclusion window
- Euclidean distance metric
- Similarity weights = 1 - d/d_max, then squared to emphsise closest match
- NaN imputed with cumulative column means

In [105]:
# ==========================================================================
# Distance model implementation
# ==========================================================================
# Steps:
#  1. Rolling z-score normalisation of each state variable
#  2. Winsorise at +/-3
#  3. For each month t, compute Euclidean distance to all historical months
#  4. Exclude the most recent `exclusion_window` months
#  5. Convert distances to similarity weights 

def fit_distance_model(state_df, norm_window=120, min_periods=24, winsorize_limit=3.0, exclusion_window=36):
    # 1. Rolling z-score
    if np.isinf(norm_window):
        rolling_mean = state_df.expanding(min_periods=min_periods).mean()
        rolling_std = state_df.expanding(min_periods=min_periods).std()
    else:
        rolling_mean = state_df.rolling(norm_window, min_periods=min_periods).mean()
        rolling_std = state_df.rolling(norm_window, min_periods=min_periods).std()
    
    normalised = (state_df - rolling_mean) / rolling_std

    # 2. Winsorize
    if winsorize_limit is not None:
        normalised = normalised.clip(-winsorize_limit, winsorize_limit)

    # 3-5. Compute distance and weights
    dates = normalised.index
    n = len(dates)
    factors_np = normalised.values  # (n, n_vars)
    weights_matrix = pd.DataFrame(np.nan, index=dates, columns=dates)

    # Accumulate cleaned historical data incrementally
    cumulative_data = []
    cumulative_date_indices = []

    for i in range(n):
        current_row = factors_np[i, :]

        # Skip if current row is all NaN
        if np.isnan(current_row).all():
            continue

        # Need at least 2 accumulated rows before computing distances
        if len(cumulative_data) < 2:
            current_filled = np.nan_to_num(current_row, nan=0.0)
            cumulative_data.append(current_filled)
            cumulative_date_indices.append(i)
            continue

        # Impute NaN in current row with cumulative column means
        cum_array = np.array(cumulative_data)
        col_means = np.nanmean(cum_array, axis=0)
        col_means = np.nan_to_num(col_means, nan=0.0)
        current_filled = np.where(np.isnan(current_row), col_means, current_row)

        # Apply exclusion window
        if exclusion_window > 0:
            eligible_mask = np.array(cumulative_date_indices) <= (i - exclusion_window - 1)
        else:
            eligible_mask = np.ones(len(cumulative_date_indices), dtype=bool)

        if eligible_mask.sum() < 2:
            cumulative_data.append(current_filled)
            cumulative_date_indices.append(i)
            continue

        eligible_array = cum_array[eligible_mask]
        eligible_indices = np.array(cumulative_date_indices)[eligible_mask]

        distances = np.sqrt(np.sum((eligible_array - current_filled) ** 2, axis=1))

        # Convert to similarity
        max_dist = distances.max() if len(distances) > 0 else 1.0
        if max_dist > 0:
            similarities = 1 - distances / max_dist
        else:
            similarities = np.ones_like(distances) 

        # Normalise to max=1
        max_sim = similarities.max()
        if max_sim > 0:
            similarities /= max_sim
        else:
            similarities = np.ones_like(similarities) # If all similarities are zero, treat as perfect similarity

        # Square transform (emphsise most similar periods)
        similarities = similarities ** 2

        # Normalise to sum to 1
        sim_sum = similarities.sum()
        if sim_sum > 0:
            similarities = similarities / sim_sum

        for k, hist_idx in enumerate(eligible_indices):
            weights_matrix.iloc[i, hist_idx] = similarities[k]

        # Add current row to cumulative data
        cumulative_data.append(current_filled)
        cumulative_date_indices.append(i)

    n_valid = weights_matrix.notna().any(axis=1).sum()
    print(f'Distance model fitted: {n_valid} valid periods out of {n}')
    print(f'Variables: {list(state_df.columns)}')

    return weights_matrix, normalised

# Fit model
state_aligned_filtered = state_aligned[['oil', 'copper', 'vix', 'yield_curve', 'spx', 'stock_bond_corr', 'us_3m']]
weights_matrix, normalised_state = fit_distance_model(state_aligned_filtered, **DISTANCE_CONFIG)

Distance model fitted: 153 valid periods out of 214
Variables: ['oil', 'copper', 'vix', 'yield_curve', 'spx', 'stock_bond_corr', 'us_3m']


## 4. Similarity Analysis

Examine the similarity weights to understand which historical periods are most similar to recent conditions

In [106]:
# Visualise weights for the most recent month
latest_date = weights_matrix.index[-1]
latest_weights = weights_matrix.loc[latest_date].dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=latest_weights.index,
    y=latest_weights.values,
    marker_color=COLOURS[0]
))
fig.update_layout(
    title=f'Similarity Weights: {latest_date.strftime('%Y-%m')} vs All Historical Periods',
    xaxis_title='Historical date',
    yaxis_title='Similarity weight',
    height=400,
    template='plotly_white'
)
fig.show()

# Top 5 most similar periods
top_similar = latest_weights.nlargest(10)
print(f'\nTop 10 most similar historical periods to {latest_date.strftime('%Y-%m')}')
for date, weight in top_similar.items():
    print(f'   {date.strftime('%Y-%m')}: {weight:.2f}')


Top 10 most similar historical periods to 2026-04
   2022-03: 0.01
   2014-06: 0.01
   2016-11: 0.01
   2016-12: 0.01
   2014-05: 0.01
   2011-06: 0.01
   2017-02: 0.01
   2011-07: 0.01
   2017-03: 0.01
   2011-08: 0.01


## 5. Sector Rotation Backtest

Three approaches
1. **Top-K rotation**: Each month, overweight the K sectors with the highest similarity-weighted forecast excess return, equal-weight the rest at zero. Pure long-only rotation.
2. **Long-short rotation**: Long top K, short bottom K sectors (dollar neutral).
3. **Continuous weights**: Use similarity-weighted average returns to forecast, then scale by forecast variability

In [107]:
# ==========================================================================
# Generate sector forecasts
# ==========================================================================

def forecast_sector_returns(weights_matrix, excess_returns):
    """
    For each month t, comppute similarity-weighted average of historical
    sector excess returns as the forecast.
    """
    common = weights_matrix.index.intersection(excess_returns.index)
    forecasts = pd.DataFrame(np.nan, index=common, columns=excess_returns.columns)

    for t in common:
        w = weights_matrix.loc[t].dropna()
        if len(w) == 0:
            continue
        hist_dates = w.index.intersection(excess_returns.index)
        if len(hist_dates) == 0:
            continue
        w_aligned = w[hist_dates]
        w_aligned = w_aligned / w_aligned.sum()

        for col in excess_returns.columns:
            vals = excess_returns.loc[hist_dates, col].dropna()
            common_hist = vals.index.intersection(w_aligned.index)
            if len(common_hist) > 0:
                w_sub = w_aligned[common_hist]
                w_sub = w_sub / w_sub.sum()
                forecasts.loc[t, col] = (vals[common_hist] * w_sub).sum()

    return forecasts

forecasts = forecast_sector_returns(weights_matrix, excess_aligned)

valid_forecasts = forecasts.dropna(how='all')
print(f'Sector forecasts generated: {valid_forecasts.shape}')
print(f'Date range: {valid_forecasts.index.min().date()} to {valid_forecasts.index.max().date()}')

# Latest forecast
latest_fc = valid_forecasts.iloc[-1].dropna().sort_values(ascending=False)
print(f'\nLatest forecast ({valid_forecasts.index[-1].strftime("%Y-%m")}):')
for ticker, val in latest_fc.items():
    label = SECTOR_ETFS.get(ticker, ticker)
    print(f'   {ticker:5s} ({label:16s}): {val*100:+.2f}%')

Sector forecasts generated: (153, 13)
Date range: 2013-08-31 to 2026-04-30

Latest forecast (2026-04):
   XLK   (Technology      ): +0.60%
   XLV   (Health Care     ): +0.37%
   XLY   (Consumer Disc.  ): +0.32%
   XLE   (Energy          ): +0.26%
   XLI   (Industrials     ): +0.22%
   XLF   (Financials      ): +0.20%
   XLB   (Materials       ): +0.08%
   XLU   (Utilities       ): +0.08%
   XLP   (Consumer Staples): -0.00%
   XLRE  (Real Estate     ): -0.25%
   XLC   (Comm. Services  ): -0.79%
   GLD   (Gold            ): -0.83%
   TLT   (LT Bond         ): -0.95%


In [108]:
# ==========================================================================
# Portfolio consturction and backtest
# ==========================================================================

TOP_K = 5   # Number of sectors to overweight / underweight

def calc_perf(returns, name):
    returns = returns.dropna()
    if len(returns) < 12:
        return {
            'Strategy': name,
            'Ann. Return (%)': np.nan,
            'Ann. Vol (%)': np.nan,
            'Sharpe': np.nan,
            'Hit Rate': np.nan,
            'Max DD (%)': np.nan,
            'Months': 0
        }
    
    ann_ret = returns.mean() * 12
    ann_vol = returns.std() * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    cum = returns.cumsum()
    max_dd = (cum - cum.cummax()).min()
    hit_rate = (returns > 0).mean()

    return {
        'Strategy': name,
        'Ann. Return (%)': ann_ret,
        'Ann. Vol (%)': ann_vol,
        'Sharpe': sharpe,
        'Hit Rate': hit_rate,
        'Max DD (%)': max_dd,
        'Months': len(returns)
    }

# --- Top-K rotation ---
topk_returns = pd.Series(dtype=float)

for t in forecasts.index:
    fc = forecasts.loc[t].dropna()
    if len(fc) < TOP_K:
        continue

    # Rank sector by forecast then pick top k
    top_sectors = fc.nlargest(TOP_K).index.tolist()
    # Next month's return for those sectors (actual)
    next_idx = sector_aligned.index.get_indexer([t], method='pad')[0] + 1
    if next_idx >= len(sector_aligned):
        continue
    next_date = sector_aligned.index[next_idx]
    port_ret = sector_aligned.loc[next_date, top_sectors].mean()
    topk_returns.loc[next_date] = port_ret

topk_returns = topk_returns.dropna()

# --- Long short K sectors ---
ls_returns = pd.Series(dtype=float)

for t in forecasts.index:
    fc = forecasts.loc[t].dropna()
    if len(fc) < 2 * TOP_K:
        continue

    top_sectors = fc.nlargest(TOP_K).index.tolist()
    bottom_sectors = fc.nsmallest(TOP_K).index.tolist()
    # Next month's return for those sectors (actual)
    next_idx = sector_aligned.index.get_indexer([t], method='pad')[0] + 1
    if next_idx >= len(sector_aligned):
        continue
    next_date = sector_aligned.index[next_idx]
    long_ret = sector_aligned.loc[next_date, top_sectors].mean()
    short_ret = sector_aligned.loc[next_date, bottom_sectors].mean()
    ls_returns.loc[next_date] = long_ret - short_ret

ls_returns = ls_returns.dropna()

# --- Continuous tilt around equal weight ---
cont_returns = pd.Series(dtype=float)

for t in forecasts.index:
    fc = forecasts.loc[t].dropna()
    if len(fc) < 5:
        continue
    # Scale forecasts to get tilts: z-score across sectors, then apply as weight offset
    fc_z = (fc - fc.mean()) / fc.std() if fc.std() > 0 else fc * 0
    # Base weight = 1/N, tilt = base * (1 + lambda * z)
    n = len(fc)
    base_w = 1.0 / n
    tilt_lambda = 1.0   # strength of tilt
    weights = base_w * (1 + tilt_lambda * fc_z)
    weights = weights.clip(lower=0)     # long only
    weights = weights / weights.sum()

    # Next month's return for those sectors (actual)
    next_idx = sector_aligned.index.get_indexer([t], method='pad')[0] + 1
    if next_idx >= len(sector_aligned):
        continue
    next_date = sector_aligned.index[next_idx]
    
    rets = sector_aligned.loc[next_date, fc.index].dropna()
    common_tickers = weights.index.intersection(rets.index)
    w_sub = weights[common_tickers]
    w_sub = w_sub / w_sub.sum()
    cont_returns.loc[next_date] = (rets[common_tickers] * w_sub).sum()

cont_returns = cont_returns.dropna()

# --- Continuous weight ---
cont_returns_2 = pd.Series(dtype=float)

for t in forecasts.index:
    fc = forecasts.loc[t].dropna()
    if len(fc) < 5:
        continue
    weights = fc.clip(lower=0)     # long only
    # weights = fc

    # Next month's return for those sectors (actual)
    next_idx = sector_aligned.index.get_indexer([t], method='pad')[0] + 1
    if next_idx >= len(sector_aligned):
        continue
    next_date = sector_aligned.index[next_idx]
    
    rets = sector_aligned.loc[next_date, fc.index].dropna()
    common_tickers = weights.index.intersection(rets.index)
    w_sub = weights[common_tickers]
    w_sub = w_sub / w_sub.sum()
    cont_returns_2.loc[next_date] = (rets[common_tickers] * w_sub).sum()

cont_returns_2 = cont_returns_2.dropna()

# --- Benchmarks ---
# Equal weighted
ew_bench = sector_aligned.mean(axis=1).dropna()

# SPY
spy_aligned = spy_returns.reindex(sector_aligned.index)

# Align all to common start
all_starts = [s.index.min() for s in [topk_returns, ls_returns, cont_returns] if len(s) > 0]
common_start = max(all_starts) if all_starts else ew_bench.index.min()

perf_table = pd.DataFrame([
    calc_perf(spy_aligned[common_start:], 'SPY (market)'),
    calc_perf(ew_bench[common_start:], 'Equal Weight Sector'),
    calc_perf(topk_returns[common_start:], f'Top-{TOP_K} Rotation'),
    calc_perf(ls_returns[common_start:], f'Long Short (Top/Bottom {TOP_K})'),
    calc_perf(cont_returns[common_start:], f'Continuous Tilt'),
    calc_perf(cont_returns_2[common_start:], f'Continuous Weight')
])

print(f'Sector Rotation Performance (from {common_start.strftime("%Y-%m")}):')
print(perf_table.to_string(index=False))
    

Sector Rotation Performance (from 2013-09):
                 Strategy  Ann. Return (%)  Ann. Vol (%)  Sharpe  Hit Rate  Max DD (%)  Months
             SPY (market)           0.1380        0.1428  0.9664    0.6974     -0.2538     152
      Equal Weight Sector           0.1125        0.1212  0.9281    0.6842     -0.1912     152
           Top-5 Rotation           0.1187        0.1283  0.9248    0.6776     -0.2600     152
Long Short (Top/Bottom 5)           0.0075        0.0993  0.0760    0.4934     -0.1739     152
          Continuous Tilt           0.1204        0.1208  0.9968    0.6974     -0.2080     152
        Continuous Weight           0.1154        0.1283  0.8992    0.6776     -0.2956     152


In [109]:
# Cumulative returns comparison
fig = go.Figure()

start = common_start

for returns, name, color in [
    (spy_aligned[start:], 'SPY (market)', COLOURS[0]),
    (ew_bench[start:], 'Equal Weight Sector', COLOURS[1]),
    (topk_returns[start:], f'Top-{TOP_K} Rotation', COLOURS[2]),
    (ls_returns[start:], f'Long Short (Top/Bottom {TOP_K})', COLOURS[3]),
    (cont_returns[start:], f'Continuous Tilt', COLOURS[4]),
    (cont_returns_2[start:], f'Continuous Weight', COLOURS[5])
]:
    if len(returns) == 0:
        continue

    cum = (1 + returns).cumprod() - 1
    fig.add_trace(go.Scatter(
        x=cum.index, y=cum.values * 100,
        name=name, line=dict(color=color, width=2)
    ))

fig.update_layout(
    title='Distance Model Sector Rotation: Cumulative Returns',
    xaxis_title='Date',
    yaxis_title='Cumulative Return (%)',
    height=500,
    template='plotly_white',
    legend=dict(font=dict(size=10))
)
fig.show()

In [110]:
# Monthly heatmap (continuous timing)
# Rows = years, columns = months
cont_df = cont_returns.dropna().copy()
cont_df = cont_df[cont_df != 0]
cont_df.index = pd.to_datetime(cont_df.index)

heatmap_df = pd.DataFrame({
    'year': cont_df.index.year,
    'month': cont_df.index.month,
    'return': cont_df.values * 100
})
heatmap_pivot = heatmap_df.pivot_table(index='year', columns='month', values='return', aggfunc='sum')

# Reindex to full 12 month grid
all_years = range(heatmap_pivot.index.min(), heatmap_pivot.index.max() + 1)
heatmap_pivot = heatmap_pivot.reindex(index=all_years, columns=range(1, 13))
heatmap_pivot.columns = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                         'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

text_vals = np.where(
    heatmap_pivot.notna(),
    heatmap_pivot.round(1).astype(object).fillna('').astype(str),
    ''
)

fig = go.Figure(data=go.Heatmap(
    z=heatmap_pivot.values,
    x=heatmap_pivot.columns.tolist(),
    y=[str(y) for y in heatmap_pivot.index.tolist()],
    colorscale=[
        [0.0, '#de1a24'],
        [0.5, '#FFFFFF'],
        [1.0, '#056517']
    ],
    zmid=0,
    text=text_vals,
    texttemplate='%{text}',
    textfont=dict(size=9),
    colorbar=dict(title='Return (%)'),
    xgap=1,
    ygap=1
))

fig.update_layout(
    title='Continuous Tilt: Monthly Return Heatmap (%)',
    xaxis_title='Month',
    yaxis_title='Year',
    yaxis=dict(autorange='reversed', dtick=1, type='category'),
    height=max(800, len(heatmap_pivot) * 22),
    width=1000,
    template='plotly_white',
    font=dict(size=10)
)

fig.show()

# Annual return bar chart - all 3 strategies
annual_all = pd.DataFrame()
for returns, name in [
    (spy_aligned[start:], 'SPY (market)'),
    (ew_bench[start:], 'Equal Weight Sector'),
    (topk_returns[start:], f'Top-{TOP_K} Rotation'),
    (ls_returns[start:], f'Long Short (Top/Bottom {TOP_K})'),
    (cont_returns[start:], f'Continuous Tilt')
]:
    r = returns.dropna() * 100
    r.index = pd.to_datetime(r.index)
    yr = r.groupby(r.index.year).sum()
    annual_all[name] = yr

years = [str(y) for y in annual_all.index]

fig2 = go.Figure()
for name, color in [
    ('SPY (market)', COLOURS[0]),
    ('Equal Weight Sector', COLOURS[1]),
    (f'Top-{TOP_K} Rotation', COLOURS[2]),
    (f'Long Short (Top/Bottom {TOP_K})', COLOURS[3]),
    (f'Continuous Tilt', COLOURS[4]),
]:
    fig2.add_trace(go.Bar(
        x=years,
        y=annual_all[name].values,
        name=name,
        marker_color=color
    ))

fig2.update_layout(
    title='Annual Returns by Strategy (%)',
    xaxis_title='Year',
    yaxis_title='Return (%)',
    barmode='group',
    height=450,
    template='plotly_white',
    font=dict(size=10),
    xaxis=dict(tickangle=-45, tickfont=dict(size=8)),
    legend=dict(font=dict(size=10))
)

fig2.show()

## 6. Per Sector Timing Analysis

Examine which individual sectors benefit most from the timing signal

In [111]:
# ==========================================================================
# Per sector analysis
# ==========================================================================
# For each sector, measure the correlation between the distance model forecast
# and actual next-month excess return.

sector_perf = []
for sector in excess_aligned.columns:
    fc = forecasts[sector].dropna()
    actual = excess_aligned[sector].shift(-1).dropna()  # next month's excess return
    common_idx = fc.index.intersection(actual.index)

    if len(common_idx) < 24:
        continue

    fc_aligned = fc[common_idx]
    actual_aligned = actual[common_idx]

    # Information Coefficient (rank correlation)
    ic = fc_aligned.corr(actual_aligned, method='spearman')

    # Directional accuracy: does the sign of the forecast match the sign of actual?
    dir_acc = ((fc_aligned > 0) == (actual_aligned > 0)).mean()

    # Static Sharpe (buy and hold excess return)
    static_xs = excess_aligned[sector].dropna()
    static_sharpe = (static_xs.mean() / static_xs.std()) * np.sqrt(12) if static_xs.std() > 0 else 0

    label = SECTOR_ETFS.get(sector, sector)

    sector_perf.append({
        'ETF': sector,
        'Sector': label,
        'IC (Spearman)': round(ic, 3),
        'Directional Accuracy': round(dir_acc, 3),
        'Static Sharpe': round(static_sharpe, 3),
        'Months': len(common_idx)
    })

sector_perf_df = pd.DataFrame(sector_perf).sort_values('IC (Spearman)', ascending=False)
print('Per Sector Forecast Quality:')
print(sector_perf_df.to_string(index=False))

avg_ic = sector_perf_df['IC (Spearman)'].mean()
avg_da = sector_perf_df['Directional Accuracy'].mean()
print(f'\nAverage IC: {avg_ic:.3f}')
print(f'Average directional accuracy: {avg_da:.3f}')

Per Sector Forecast Quality:
 ETF           Sector  IC (Spearman)  Directional Accuracy  Static Sharpe  Months
 XLC   Comm. Services         0.3850                0.5360         0.1160      56
 XLF       Financials         0.0440                0.4740         0.0100     152
 GLD             Gold         0.0390                0.5200        -0.0200     152
 XLE           Energy         0.0280                0.4610        -0.1000     152
 XLP Consumer Staples         0.0270                0.5000        -0.0800     152
 XLB        Materials         0.0270                0.5070        -0.1150     152
 TLT          LT Bond        -0.0130                0.5660        -0.3650     152
 XLU        Utilities        -0.0330                0.5070        -0.1030     152
 XLI      Industrials        -0.0390                0.5070         0.2440     152
 XLK       Technology        -0.0400                0.5390         0.5940     152
 XLY   Consumer Disc.        -0.0670                0.5260         0.

In [112]:
# Chart: Static vs Timed Sharpe by Factor
fig = go.Figure()

sp = sector_perf_df.sort_values('IC (Spearman)', ascending=True)
colors = [COLOURS[0] if v > 0 else COLOURS[1] for v in sp['IC (Spearman)']]

fig.add_trace(go.Bar(
    y=[f"{row['ETF']} ({row['Sector']})" for _, row in sp.iterrows()],
    x=sp['IC (Spearman)'],
    orientation='h',
    marker_color=colors
))
fig.add_vline(x=0, line_dash='dash')

fig.update_layout(
    title='Information Coefficient by Sector (Forecast vs Next-Month Return)',
    xaxis_title='IC (Spearman)',
    height=max(500, len(sp) * 30),
    template='plotly_white'
)
fig.show()

## 7. State Variable Cross Correlations

Verify that the state variables provide independent information

In [113]:
# State variable correlation matrix
state_corr = state_aligned_filtered.corr()

mask = np.triu(np.ones(state_corr.shape, dtype=bool), k=0)
state_corr_masked = state_corr.where(~mask)

labels = [str(c)[:20] for c in state_corr.columns]
# Text array with empty string for masked cells
text_vals = np.where(mask, '', state_corr.round(2).astype(str).values)

fig = go.Figure(data=go.Heatmap(
    z=state_corr_masked.values,
    x=labels,
    y=labels,
    colorscale='RdBu_r',
    # colorscale='RdYlGn',
    zmin=-1, zmax=1,
    showscale=True,
    text=text_vals,
    texttemplate='%{text}',
    textfont=dict(size=9, color='black')
))
fig.update_layout(
    title='State Variable Cross Correlations (12M changes)',
    height=max(700, len(state_corr) * 50),
    width=max(700, len(state_corr) * 50),
    font=dict(size=9)
)
fig.show()

# Average absolute correlation
upper_tri = state_corr.where(np.triu(np.ones(state_corr.shape), k=1).astype(bool))
avg_abs = upper_tri.abs().stack().mean()
print(f'\nAverage absolute cross correlation: {avg_abs:.2f}')


Average absolute cross correlation: 0.30


### 8. Current Regime Sector Allocation

Model's current recommended sector allocation

In [ ]:
# ==========================================================================
# Current allocation snapshot
# ==========================================================================

latest_fc = forecasts.iloc[-1].dropna().sort_values(ascending=False)
latest_date = forecasts.index[-1]

# Continuous tilt weights
fc_z = (latest_fc - latest_fc.mean()) / latest_fc.std() if latest_fc.std() > 0 else latest_fc * 0
n = len(latest_fc)
base_w = 1.0 / n
tilt_lambda = 0.5
weights_current = base_w * (1 + tilt_lambda * fc_z)
weights_current = weights_current.clip(lower=0)
weights_current = weights_current/ weights_current.sum()

alloc = pd.DataFrame({
    'ETF': latest_fc.index,
    'Sector': [SECTOR_ETFS.get(t, t) for t in latest_fc.index],
    'Forecast XS (bps)': (latest_fc.values * 10000).round(1),
    'Model Weight (%)': (weights_current[latest_fc.index].values * 100).round(1),
    'EW Weight (%)': round(100 / n, 1),
    'Active Tilt (%)': ((weights_current[latest_fc.index].values - 1/n) * 100).round(1)
})

print(f'Current Sector Allocation - {latest_date.strftime("%Y-%m")}')
print('='*70)
print(alloc.to_string(index=False))

# Top and bottom picks
top_n = min(5, len(latest_fc))
print(f'\n Top {top_n} overweights: {", ".join(latest_fc.head(top_n).index)}')
print(f'Top {top_n} underweights: {", ".join(latest_fc.tail(top_n).index)}')

# Chart
fig = go.Figure()
sorted_alloc = alloc.sort_values('Active Tilt (%)')
colors = [COLOURS[0] if v > 0 else COLOURS[1] for v in sp['IC (Spearman)']]

fig.add_trace(go.Bar(
    y=[f"{row['ETF']} ({row['Sector']})" for _, row in sorted_alloc.iterrows()],
    x=sorted_alloc['Active Tilt (%)'],
    orientation='h',
    marker_color=colors,
    text=[f"{v:+.1f}%" for v in sorted_alloc['Active Tilt (%)']],
    textposition='outside'
))

fig.add_vline(x=0, line_dash='dash', line_color='grey')
fig.update_layout(
    title=f'Active Sector Tilts vs Equal-Weight - {latest_date.strftime("%Y-%m")}',
    xaxis_title='Active Weight (%)',
    height=max(500, len(sorted_alloc) * 30),
    template='plotly_white',
    margin=dict(l=180)
)
fig.show()

Current Sector Allocation - 2026-04
 ETF           Sector  Forecast XS (bps)  Model Weight (%)  EW Weight (%)  Active Tilt (%)
 XLK       Technology            59.6000           12.7000         7.7000           5.0000
 XLV      Health Care            37.1000           11.0000         7.7000           3.3000
 XLY   Consumer Disc.            32.2000           10.6000         7.7000           2.9000
 XLE           Energy            25.6000           10.1000         7.7000           2.4000
 XLI      Industrials            22.1000            9.8000         7.7000           2.1000
 XLF       Financials            19.6000            9.6000         7.7000           1.9000
 XLB        Materials             8.2000            8.7000         7.7000           1.0000
 XLU        Utilities             7.9000            8.7000         7.7000           1.0000
 XLP Consumer Staples            -0.5000            8.1000         7.7000           0.4000
XLRE      Real Estate           -25.0000            6.